# MinIO / S3 Parquet write

This notebook demonstrates **writing** Parquet data to **MinIO** (S3-compatible storage), as a companion to `20_minio_s3_parquet.ipynb` (which covers reading). Credentials are loaded from `.env.local`.

| # | Topic |
|---|---|
| 1 | Setup — load credentials, configure SSRF allowlist, test connectivity |
| 2 | Seed a local source table |
| 3 | `ParquetSink` — write a DataFrame directly to S3 |
| 4 | `ParquetPipeline` — SQL source → S3, with reload |
| 5 | Cleanup |


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti_data").exists():
    parent = PROJECT_ROOT.parent.resolve()
    if (parent / "src" / "boti_data").exists():
        PROJECT_ROOT = parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import datetime as dt
import os
import shutil
import tempfile

import pandas as pd
from sqlalchemy import Date, Integer, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti_data import DataHelper, ParquetPipeline, ParquetReader, ParquetSink
from boti_data.connection_catalog import S3Catalog


## 1. Setup — load credentials, configure SSRF allowlist, test connectivity

The ETL credentials live in `.env.local` (repo root, git-ignored). The MinIO endpoint is on a private network, so we add the host to the SSRF allowlist.


In [2]:
# Verify the .env.local file exists
env_file = PROJECT_ROOT / ".env.local"
print(f".env.local exists: {env_file.exists()}")
print(f".env.local path: {env_file}")


.env.local exists: True
.env.local path: /Users/lvalverdeb/TeamDev/repo-split/boti-data/.env.local


In [3]:
# Add the MinIO private IP to the SSRF allowlist so FilesystemConfig accepts it
import boti.core.filesystem as fsmod

MINIO_HOST = "10.211.55.36"
fsmod.ENDPOINT_ALLOWLIST.add(MINIO_HOST)
print(f"Allowlisted {MINIO_HOST}")
print(f"Allowlist now contains: {fsmod.ENDPOINT_ALLOWLIST}")


Allowlisted 10.211.55.36
Allowlist now contains: {'10.211.55.36'}


In [4]:
# Connectivity test — reach MinIO and make sure the bucket exists (a write
# notebook needs the bucket up front, unlike a read-only one).
from boti.core import create_filesystem

try:
    config = fsmod.FilesystemConfig.from_env_prefix("ETL_", env_file=PROJECT_ROOT / ".env.local")
    fs = create_filesystem(config)
    try:
        items = fs.ls(config.fs_path)
    except FileNotFoundError:
        print(f"Bucket '{config.fs_path}' does not exist yet — creating it")
        fs.mkdir(config.fs_path)
        items = fs.ls(config.fs_path)
    print(f"Connected to MinIO. Bucket '{config.fs_path}' contains {len(items)} top-level items.")
    CONNECTED = True
except Exception as exc:
    print(f"Could not connect to MinIO: {exc}")
    CONNECTED = False

SCRATCH_PREFIX = f"{config.fs_path if CONNECTED else 'dst-etl'}/scratch/boti_data_write_example"
print(f"Scratch prefix for this notebook's demo writes: {SCRATCH_PREFIX}")


Connected to MinIO. Bucket 'dst-etl' contains 0 top-level items.
Scratch prefix for this notebook's demo writes: dst-etl/scratch/boti_data_write_example


## 2. Seed a local source table

A tiny local SQLite table stands in for a real upstream source. We will use this both as a plain pandas frame (section 3) and as a `DataHelper`-backed SQL source (section 4).


In [5]:
class Base(DeclarativeBase):
    pass


class Event(Base):
    __tablename__ = "events"

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(16))


tmp_dir = tempfile.mkdtemp(prefix="boti_data_write_example_")
db_path = Path(tmp_dir) / "write_example.db"
sqlite_dsn = f"sqlite:///{db_path}"
WORKER_DSN_ENV_VAR = "BOTI_EXAMPLE_MINIO_WRITE_SQLITE_DSN"

engine = create_engine(sqlite_dsn)
try:
    Base.metadata.create_all(engine)
    with Session(engine) as session:
        session.add_all(
            [
                Event(id=1, event_date=dt.date(2026, 1, 2), status="ok"),
                Event(id=2, event_date=dt.date(2026, 1, 2), status="ok"),
                Event(id=3, event_date=dt.date(2026, 1, 3), status="error"),
            ]
        )
        session.commit()
finally:
    engine.dispose()

frame = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "event_date": [dt.date(2026, 1, 2), dt.date(2026, 1, 2), dt.date(2026, 1, 3)],
        "status": ["ok", "ok", "error"],
    }
)
print(f"Local SQLite source: {db_path}")
print(f"Seeded {frame.shape[0]} rows")


Local SQLite source: /var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_data_write_example_xupz1wrm/write_example.db
Seeded 3 rows


## 3. `ParquetSink` — write a DataFrame directly to S3

For a one-off write of an in-memory frame (no SQL source needed), `ParquetSink` writes directly. The destination is built the same way as the read notebook: `S3Catalog` supplies the filesystem, wrapped in a `ParquetReader(config, fs=fs)`.


In [6]:
if CONNECTED:
    s3 = S3Catalog("ETL_", env_file=PROJECT_ROOT / ".env.local")
    fs = s3.fs()

    sink_path = f"{SCRATCH_PREFIX}/direct_sink"
    with ParquetReader({"parquet_storage_path": sink_path}, fs=fs) as destination:
        with ParquetSink(destination, partition_on=("partition_date",)) as sink:
            result = sink.write(frame, date_field="event_date")

    print(f"Wrote to: {result.path}")
    print("Files:")
    for file_path in result.files:
        print(f"  {file_path}")
else:
    print("Skipping — MinIO not available")


Wrote to: dst-etl/scratch/boti_data_write_example/direct_sink
Files:
  dst-etl/scratch/boti_data_write_example/direct_sink/partition_date=2026-01-02
  dst-etl/scratch/boti_data_write_example/direct_sink/partition_date=2026-01-03


## 4. `ParquetPipeline` — SQL source → S3, with reload

`ParquetPipeline` wraps a source (a `DataHelper`, here backed by the SQLite table) and a `ParquetSink`, exposing `materialize()` to write and — with `reload=True` — read the written data straight back in one call.

`worker_connection_env_var` is set on the `DataHelper` so the raw DSN is not embedded in the pipeline; without it, `WorkerSqlConfig` logs a warning about credentials being visible in scheduler logs.


In [7]:
if CONNECTED:
    previous_worker_dsn = os.environ.get(WORKER_DSN_ENV_VAR)
    os.environ[WORKER_DSN_ENV_VAR] = sqlite_dsn
    try:
        helper = DataHelper(
            backend="sqlalchemy",
            connection_url=sqlite_dsn,
            worker_connection_env_var=WORKER_DSN_ENV_VAR,
            poolclass="sqlalchemy.pool.NullPool",
            query_only=False,
            table="events",
        )
        s3 = S3Catalog("ETL_", env_file=PROJECT_ROOT / ".env.local")
        fs = s3.fs()
        pipeline_path = f"{SCRATCH_PREFIX}/pipeline"
        destination = ParquetReader({"parquet_storage_path": pipeline_path}, fs=fs)
        with ParquetPipeline(
            helper, destination, date_field="event_date", partition_on=("partition_date",)
        ) as pipeline:
            materialized = pipeline.materialize(
                reload=True, reload_options={"return_type": "pandas"}
            )
    finally:
        if previous_worker_dsn is None:
            os.environ.pop(WORKER_DSN_ENV_VAR, None)
        else:
            os.environ[WORKER_DSN_ENV_VAR] = previous_worker_dsn

    print(f"Wrote to: {materialized.path}")
    print(f"Reloaded rows: {len(materialized.frame)}")
    print(materialized.frame.sort_values("id").reset_index(drop=True))
else:
    print("Skipping — MinIO not available")


Wrote to: dst-etl/scratch/boti_data_write_example/pipeline
Reloaded rows: 3
   id                event_date status
0   1 2026-01-02 00:00:00+00:00     ok
1   2 2026-01-02 00:00:00+00:00     ok
2   3 2026-01-03 00:00:00+00:00  error


## 5. Cleanup

Remove everything this notebook wrote under its scratch prefix, and the local temp SQLite database, so re-running it against the shared MinIO instance does not accumulate objects.


In [8]:
if CONNECTED:
    if fs.exists(SCRATCH_PREFIX):
        fs.rm(SCRATCH_PREFIX, recursive=True)
        print(f"Removed {SCRATCH_PREFIX} from MinIO")
    else:
        print(f"{SCRATCH_PREFIX} already absent")
else:
    print("Skipping — MinIO not available")

shutil.rmtree(tmp_dir, ignore_errors=True)
print("Removed local temp SQLite database")


Removed dst-etl/scratch/boti_data_write_example from MinIO
Removed local temp SQLite database


### Summary

- **`ParquetSink`** — write a single in-memory frame directly to S3; wrap a `ParquetReader(config, fs=fs)` as the destination.
- **`ParquetPipeline`** — SQL source (`DataHelper`) → S3 sink, with `materialize(reload=True)` writing and reading back in one call.
- **`worker_connection_env_var`** — set on any `DataHelper` used as a write source to avoid the `WorkerSqlConfig` warning about raw DSNs in worker/scheduler logs.
- **Cleanup** — this notebook writes under a `scratch/` prefix and removes it at the end, so re-running it does not accumulate objects in the shared MinIO bucket.
- The `ConnectionCatalog` + `filesystem_profile` pattern from the read notebook does not currently extend to writes — `ParquetReader`/`ParquetSink`/`ParquetPipeline` resolve `filesystem_profile` only through a `catalog`, which they don't accept; only the read-only `ParquetDataResource` does. Use `S3Catalog` (or any pre-built `fs=`) for writes today.
